This notebook demonstrates a complete end-to-end Computer Vision pipeline using the FastAI library. It includes automated data collection via DuckDuckGo, image preprocessing, and fine-tuning a ResNet18 model to classify astronomical images (Spiral Galaxies vs. Elliptical Galaxies).

In [ ]:
import warnings
import logging
import time
from fastcore.all import *
from fastdownload import download_url
from fastai.vision.all import *


Install and import search tool

In [ ]:
!pip install -Uqq ddgs
from ddgs import DDGS

def search_images(keywords, max_images=200): 
    return L(DDGS().images(keywords, max_results=max_images)).itemgot('image')

Download the data

In [ ]:
searches = 'spiral galaxy','elliptical galaxy'
path = Path('galaxy_classifier')

for o in searches:
    dest = (path/o)
    dest.mkdir(exist_ok=True, parents=True)
    download_images(dest, urls=search_images(f'{o} hubble telescope photo'))
    time.sleep(5) 
    resize_images(path/o, max_size=400, dest=path/o)

Clean bad images

In [ ]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink)

Prepare data loaders

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=32)


Train model

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

Run a prediction

In [ ]:
test_url = search_images('spiral galaxy hubble', max_images=1)[0]
download_url(test_url, 'test.jpg', show_progress=False)
prediction,_,probs = learn.predict(PILImage.create('test.jpg'))

print(f"Result: {prediction}")
print(f"Confidence: {probs.max():.4f}")
Image.open('test.jpg').to_thumb(256,256)